---
# 🟠 PROJECT 3 — Data Cleaning Utility
**Topics:** Handle missing values, fix dtypes, parse dates, remove duplicates, standardize column names, output cleaned dataset + cleaning log

---

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
print(f"Pandas: {pd.__version__}")

Pandas: 2.2.2


## 3.1 — Create Dirty Dataset (simulating real-world messy data)

In [ ]:
dirty_csv = """ID,  Full Name ,Age,  Salary ,JoinDate,Department,Score,Status
1,Alice Johnson,29,75000.0,2019-03-15,Engineering,92.5,Active
2,Bob Smith,35,62000,22-07-2014,Marketing,,Active
3,Carol White,42,95000.0,2006/01/10,Engineering,88.0,Active
4,David Brown,abc,45000,2021-05-05,HR,76.5,Inactive
5,Eva Davis,31,82000.0,2017-09-30,Engineering,95.0,active
6,Frank Miller,38,,2012-11-18,Marketing,67.0,Active
7,Grace Wilson,25,42000.0,14/02/2022,HR,,Active
8,Henry Taylor,45,110000.0,2004-06-01,Engineering,99.0,ACTIVE
9,Iris Anderson,33,67000.0,25-08-2015,Marketing,78.5,Active
2,Bob Smith,35,62000,22-07-2014,Marketing,,Active
10,Jack Thomas,29,,2020-04-10,HR,80.0,Inactive
11,Karen Jackson,,88000.0,2012-03-20,Engineering,91.0,Active
12,Liam Harris,41,72000.0,08/10/2009,Marketing,85.5,INACTIVE
13,,28,71000.0,2020-07-17,Engineering,88.0,Active
14,Noah Martin,34,50000.0,2016-12-03,,74.0,Active
15,Olivia Garcia,30,65000.0,2018-06-28,Marketing,82.0,
8,Henry Taylor,45,110000.0,2004-06-01,Engineering,99.0,ACTIVE
"""

with open('dirty_data.csv', 'w') as f:
    f.write(dirty_csv)

raw_df = pd.read_csv('dirty_data.csv')
print("🔴 RAW DIRTY DATA:")
print(raw_df)
print(f"\nShape: {raw_df.shape}")

🔴 RAW DIRTY DATA:
    ID     Full Name   Age    Salary     JoinDate   Department  Score  \
0    1  Alice Johnson   29    75000.0  2019-03-15  Engineering   92.5   
1    2      Bob Smith   35    62000.0  22-07-2014    Marketing    NaN   
2    3    Carol White   42    95000.0  2006/01/10  Engineering   88.0   
3    4    David Brown  abc    45000.0  2021-05-05           HR   76.5   
4    5      Eva Davis   31    82000.0  2017-09-30  Engineering   95.0   
5    6   Frank Miller   38        NaN  2012-11-18    Marketing   67.0   
6    7   Grace Wilson   25    42000.0  14/02/2022           HR    NaN   
7    8   Henry Taylor   45   110000.0  2004-06-01  Engineering   99.0   
8    9  Iris Anderson   33    67000.0  25-08-2015    Marketing   78.5   
9    2      Bob Smith   35    62000.0  22-07-2014    Marketing    NaN   
10  10    Jack Thomas   29        NaN  2020-04-10           HR   80.0   
11  11  Karen Jackson  NaN    88000.0  2012-03-20  Engineering   91.0   
12  12    Liam Harris   41    720

## 3.2 — Inspect Issues in the Dirty Data

In [ ]:
print("=" * 55)
print("DATA QUALITY INSPECTION")
print("=" * 55)

print("\n📋 Column names (raw):")
print(raw_df.columns.tolist())

print("\n📊 Data Types:")
print(raw_df.dtypes)

print("\n❓ Missing Values:")
missing = raw_df.isnull().sum()
missing_pct = (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0])

print("\n🔁 Duplicate Rows:")
print(f"   Total duplicates: {raw_df.duplicated().sum()}")
print("Duplicate rows:")
print(raw_df[raw_df.duplicated(keep=False)])

print("\n📅 Inconsistent date formats in JoinDate:")
print(raw_df['JoinDate'].unique())

print("\n⚠️  Inconsistent Status values:")
print(raw_df['Status'].unique())

print("\n⚠️  Non-numeric in Age column:")
print(raw_df['Age'].unique())

DATA QUALITY INSPECTION

📋 Column names (raw):
['ID', '  Full Name ', 'Age', '  Salary ', 'JoinDate', 'Department', 'Score', 'Status']

📊 Data Types:
ID                int64
  Full Name      object
Age              object
  Salary        float64
JoinDate         object
Department       object
Score           float64
Status           object
dtype: object

❓ Missing Values:
              Missing Count  Missing %
  Full Name               1       5.88
Age                       1       5.88
  Salary                  2      11.76
Department                1       5.88
Score                     3      17.65
Status                    1       5.88

🔁 Duplicate Rows:
   Total duplicates: 2
Duplicate rows:
    ID    Full Name  Age    Salary     JoinDate   Department  Score  Status
1    2     Bob Smith  35    62000.0  22-07-2014    Marketing    NaN  Active
7    8  Henry Taylor  45   110000.0  2004-06-01  Engineering   99.0  ACTIVE
9    2     Bob Smith  35    62000.0  22-07-2014    Marketing    Na

## 3.3 — Full Data Cleaning Pipeline

In [ ]:
# ============================================================
#  DATA CLEANING UTILITY — Complete Pipeline
# ============================================================

cleaning_log = []   # collect all cleaning steps
df = raw_df.copy()  # work on a copy

# ---- STEP 1: Standardize Column Names ----
before_cols = df.columns.tolist()
df.columns = (
    df.columns
    .str.strip()           # remove leading/trailing spaces
    .str.lower()           # lowercase
    .str.replace(' ', '_') # spaces → underscores
    .str.replace(r'[^\w]', '', regex=True)  # remove special chars
)
after_cols = df.columns.tolist()
changed_cols = [(b, a) for b, a in zip(before_cols, after_cols) if b != a]
cleaning_log.append(f"[STEP 1] Standardized {len(changed_cols)} column name(s): {changed_cols}")
print("✅ STEP 1: Columns standardized")
print("  New columns:", df.columns.tolist())

# ---- STEP 2: Remove Duplicate Rows ----
dup_count = df.duplicated().sum()
df = df.drop_duplicates()
df = df.reset_index(drop=True)
cleaning_log.append(f"[STEP 2] Removed {dup_count} duplicate row(s). Rows now: {len(df)}")
print(f"\n✅ STEP 2: Removed {dup_count} duplicate(s). Remaining rows: {len(df)}")

# ---- STEP 3: Fix Incorrect dtypes ----
# Age: convert to numeric, coerce errors → NaN
bad_age_rows = df[pd.to_numeric(df['age'], errors='coerce').isna() & df['age'].notna()]
df['age'] = pd.to_numeric(df['age'], errors='coerce')
cleaning_log.append(f"[STEP 3a] Fixed Age dtype: {len(bad_age_rows)} non-numeric value(s) coerced to NaN")

# Salary: numeric
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
cleaning_log.append("[STEP 3b] Confirmed Salary as numeric (float64)")

# Score: numeric
df['score'] = pd.to_numeric(df['score'], errors='coerce')
cleaning_log.append("[STEP 3c] Confirmed Score as numeric (float64)")
print("\n✅ STEP 3: Data types fixed")

# ---- STEP 4: Parse Dates (multiple formats) ----
def parse_mixed_dates(date_series):
    """Parse dates in various formats to a single standard format."""
    formats = ['%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d', '%d/%m/%Y', '%m/%d/%Y']
    parsed = pd.Series([pd.NaT] * len(date_series), index=date_series.index)
    unparsed = date_series.copy()

    for fmt in formats:
        mask = unparsed.notna()
        try:
            trial = pd.to_datetime(unparsed[mask], format=fmt, errors='coerce')
            success = trial.notna()
            parsed[unparsed[mask][success].index] = trial[success]
            unparsed[unparsed[mask][success].index] = pd.NaT
        except:
            pass
    return parsed

df['joindate'] = parse_mixed_dates(df['joindate'])
parsing_failed = df['joindate'].isna().sum()
cleaning_log.append(f"[STEP 4] Parsed JoinDate column: {parsing_failed} value(s) could not be parsed")
print("\n✅ STEP 4: Dates parsed")
print("  Sample dates:", df['joindate'].head(5).tolist())

# ---- STEP 5: Handle Missing Values ----
print("\n✅ STEP 5: Handling missing values")

# (a) Drop rows where 'full_name' is missing (critical identifier)
before = len(df)
df = df.dropna(subset=['full_name'])
dropped_name = before - len(df)
cleaning_log.append(f"[STEP 5a] Dropped {dropped_name} row(s) where full_name is missing")
print(f"  Dropped {dropped_name} row(s) with missing full_name")

# (b) Fill Age with median (robust to outliers)
age_median = df['age'].median()
age_missing = df['age'].isna().sum()
df['age'] = df['age'].fillna(age_median)
df['age'] = df['age'].astype(int)
cleaning_log.append(f"[STEP 5b] Filled {age_missing} missing Age value(s) with median: {age_median}")
print(f"  Filled {age_missing} missing Age with median ({age_median})")

# (c) Fill Salary with department mean
salary_missing = df['salary'].isna().sum()
df['salary'] = df.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.mean())
)
# if department itself is missing, fill with global mean
df['salary'] = df['salary'].fillna(df['salary'].mean())
cleaning_log.append(f"[STEP 5c] Filled {salary_missing} missing Salary value(s) with department mean")
print(f"  Filled {salary_missing} missing Salary with department mean")

# (d) Fill Score with median
score_median = df['score'].median()
score_missing = df['score'].isna().sum()
df['score'] = df['score'].fillna(score_median)
cleaning_log.append(f"[STEP 5d] Filled {score_missing} missing Score value(s) with median: {score_median}")
print(f"  Filled {score_missing} missing Score with median ({score_median})")

# (e) Fill Department with 'Unknown'
dept_missing = df['department'].isna().sum()
df['department'] = df['department'].fillna('Unknown')
cleaning_log.append(f"[STEP 5e] Filled {dept_missing} missing Department value(s) with 'Unknown'")
print(f"  Filled {dept_missing} missing Department with 'Unknown'")

# (f) Fill Status with mode
status_mode = df['status'].mode()[0] if df['status'].notna().any() else 'Active'
status_missing = df['status'].isna().sum()
df['status'] = df['status'].fillna(status_mode)
cleaning_log.append(f"[STEP 5f] Filled {status_missing} missing Status value(s) with mode: '{status_mode}'")
print(f"  Filled {status_missing} missing Status with mode ('{status_mode}')")

# ---- STEP 6: Standardize Text Values ----
df['status'] = df['status'].str.strip().str.capitalize()
df['department'] = df['department'].str.strip().str.title()
df['full_name'] = df['full_name'].str.strip().str.title()
cleaning_log.append("[STEP 6] Standardized text: Status (capitalize), Department (title), Full_Name (title)")
print("\n✅ STEP 6: Text values standardized")
print("  Status values:",     df['status'].unique())
print("  Department values:", df['department'].unique())

# ---- STEP 7: Final verification ----
cleaning_log.append(f"[STEP 7] Final dataset: {df.shape[0]} rows × {df.shape[1]} cols. Missing values remaining: {df.isnull().sum().sum()}")
print("\n✅ STEP 7: Cleaning complete!")

✅ STEP 1: Columns standardized
  New columns: ['id', 'full_name', 'age', 'salary', 'joindate', 'department', 'score', 'status']

✅ STEP 2: Removed 2 duplicate(s). Remaining rows: 15

✅ STEP 3: Data types fixed

✅ STEP 4: Dates parsed
  Sample dates: [Timestamp('2019-03-15 00:00:00'), Timestamp('2014-07-22 00:00:00'), Timestamp('2006-01-10 00:00:00'), Timestamp('2021-05-05 00:00:00'), Timestamp('2017-09-30 00:00:00')]

✅ STEP 5: Handling missing values
  Dropped 1 row(s) with missing full_name
  Filled 2 missing Age with median (33.5)
  Filled 2 missing Salary with department mean
  Filled 2 missing Score with median (83.75)
  Filled 1 missing Department with 'Unknown'
  Filled 1 missing Status with mode ('Active')

✅ STEP 6: Text values standardized
  Status values: ['Active' 'Inactive']
  Department values: ['Engineering' 'Marketing' 'Hr' 'Unknown']

✅ STEP 7: Cleaning complete!


## 3.4 — Compare: Before vs After Cleaning

In [ ]:
print("=" * 60)
print("BEFORE vs AFTER CLEANING")
print("=" * 60)
print(f"Rows    : {raw_df.shape[0]:>6}  →  {df.shape[0]}")
print(f"Columns : {raw_df.shape[1]:>6}  →  {df.shape[1]}")
print(f"Missing : {raw_df.isnull().sum().sum():>6}  →  {df.isnull().sum().sum()}")
print(f"Dupes   : {raw_df.duplicated().sum():>6}  →  {df.duplicated().sum()}")

print("\n🟢 CLEANED DATA:")
print(df.to_string())

print("\n📊 Data Types after cleaning:")
print(df.dtypes)

print("\n❓ Remaining Missing Values:")
print(df.isnull().sum())

BEFORE vs AFTER CLEANING
Rows    :     17  →  14
Columns :      8  →  8
Missing :      9  →  0
Dupes   :      2  →  0

🟢 CLEANED DATA:
    id      full_name  age         salary   joindate   department  score    status
0    1  Alice Johnson   29   75000.000000 2019-03-15  Engineering  92.50    Active
1    2      Bob Smith   35   62000.000000 2014-07-22    Marketing  83.75    Active
2    3    Carol White   42   95000.000000 2006-01-10  Engineering  88.00    Active
3    4    David Brown   33   45000.000000 2021-05-05           Hr  76.50  Inactive
4    5      Eva Davis   31   82000.000000 2017-09-30  Engineering  95.00    Active
5    6   Frank Miller   38   66500.000000 2012-11-18    Marketing  67.00    Active
6    7   Grace Wilson   25   42000.000000 2022-02-14           Hr  83.75    Active
7    8   Henry Taylor   45  110000.000000 2004-06-01  Engineering  99.00    Active
8    9  Iris Anderson   33   67000.000000 2015-08-25    Marketing  78.50    Active
9   10    Jack Thomas   29   43500.

## 3.5 — Output Cleaned Dataset + Cleaning Log

In [ ]:
# ---- Save cleaned CSV ----
df.to_csv('cleaned_data.csv', index=False)
print("✅ Cleaned data saved to cleaned_data.csv")

# ---- Save cleaned Excel ----
df.to_excel('cleaned_data.xlsx', index=False)
print("✅ Cleaned data saved to cleaned_data.xlsx")

# ---- Generate & Save Cleaning Log ----
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

log_lines = [
    "=" * 60,
    "  DATA CLEANING LOG",
    f"  Generated: {timestamp}",
    "=" * 60,
    f"  Input file  : dirty_data.csv",
    f"  Output file : cleaned_data.csv",
    f"  Input rows  : {raw_df.shape[0]}",
    f"  Output rows : {df.shape[0]}",
    f"  Rows removed: {raw_df.shape[0] - df.shape[0]}",
    "",
    "  CLEANING STEPS:",
    "-" * 60,
]

for i, entry in enumerate(cleaning_log, 1):
    log_lines.append(f"  {entry}")

log_lines += [
    "",
    "-" * 60,
    f"  STATUS: Cleaning Completed Successfully ✅",
    "=" * 60,
]

log_text = "\n".join(log_lines)

# Print log
print("\n" + log_text)

# Save log to file
with open('cleaning_log.txt', 'w') as f:
    f.write(log_text)
print("\n✅ Cleaning log saved to cleaning_log.txt")

print("\n" + "=" * 60)
print("✅ Project 3 — Data Cleaning Utility Complete!")
print("=" * 60)

✅ Cleaned data saved to cleaned_data.csv
✅ Cleaned data saved to cleaned_data.xlsx

  DATA CLEANING LOG
  Generated: 2026-04-10 04:47:06
  Input file  : dirty_data.csv
  Output file : cleaned_data.csv
  Input rows  : 17
  Output rows : 14
  Rows removed: 3

  CLEANING STEPS:
------------------------------------------------------------
  [STEP 1] Standardized 8 column name(s): [('ID', 'id'), ('  Full Name ', 'full_name'), ('Age', 'age'), ('  Salary ', 'salary'), ('JoinDate', 'joindate'), ('Department', 'department'), ('Score', 'score'), ('Status', 'status')]
  [STEP 2] Removed 2 duplicate row(s). Rows now: 15
  [STEP 3a] Fixed Age dtype: 1 non-numeric value(s) coerced to NaN
  [STEP 3b] Confirmed Salary as numeric (float64)
  [STEP 3c] Confirmed Score as numeric (float64)
  [STEP 4] Parsed JoinDate column: 0 value(s) could not be parsed
  [STEP 5a] Dropped 1 row(s) where full_name is missing
  [STEP 5b] Filled 2 missing Age value(s) with median: 33.5
  [STEP 5c] Filled 2 missing Salary 